# Isolation Forest filtered asking-price ensemble

CPU training on the complete original cohort. Isolation Forest uses contamination="auto" on training vehicle facts and frozen CarBids signals, never asking labels. Validation and test cars stay unfiltered. Eight LightGBM candidates, validation selection and three fixed seeds; no neural epochs or GPU. Full deployment refitting follows assessment. The original unfiltered ensemble is the comparison benchmark. The historical test has been reused; this is not fresh external validation. Save Kaggle outputs to resume.


In [ ]:
import sys,subprocess,os,time,json,base64,io,zipfile,hashlib,shutil,signal
from pathlib import Path
START=time.monotonic()
subprocess.run([sys.executable,'-m','pip','install','-q','lightgbm>=4.5,<5','pandas>=2.1,<3','scikit-learn>=1.4,<2','requests>=2.31,<3'],check=True)
ROOT=Path('/kaggle/working/robust-pricing');ROOT.mkdir(parents=True,exist_ok=True)


In [ ]:
SOURCE_ZIP='...'
SOURCE_SHA='5409303fe0baef8792559479944d31baaab129da9d2adab96b2598ea9d3a27cf'
raw=base64.b64decode(SOURCE_ZIP)
assert hashlib.sha256(raw).hexdigest()==SOURCE_SHA
previous=list(Path('/kaggle/input').rglob('robust-session.json'))
if len(previous)>1:raise ValueError('Attach only one previous robust session output')
if previous and not (ROOT/'robust-session.json').exists():shutil.copytree(previous[0].parent,ROOT,dirs_exist_ok=True)
state=ROOT/'robust-session.json'
if state.exists() and json.loads(state.read_text())['source_hash']!=SOURCE_SHA:raise ValueError('Previous session uses different source; use a fresh output')
with zipfile.ZipFile(io.BytesIO(raw)) as z:z.extractall(ROOT)
state.write_text(json.dumps({'source_hash':SOURCE_SHA}));os.chdir(ROOT)
DATA=ROOT/'training-cache/full-cohort.jsonl';DATA.parent.mkdir(parents=True,exist_ok=True)


## Prepare the full eligible cohort

Attach the original `full-cohort.jsonl` for an exact cohort reproduction. Otherwise attach `vehicles.csv`, or the notebook downloads the public Zenodo snapshot and runs the original complete-database preparation. Do not substitute a different cohort without recording the change. The frozen upstream model excluded known VIN overlaps found in the original Pricing cohort; that exclusion audit does not automatically cover a new dataset.

In [ ]:
import requests
attached=list(Path('/kaggle/input').rglob('full-cohort.jsonl'))
if len(attached)>1:raise ValueError('Attach only one full-cohort.jsonl')
if not DATA.exists():
    if attached:shutil.copy2(attached[0],DATA)
    else:
        archive=ROOT/'raw-vehicles.zip';csvs=list(Path('/kaggle/input').rglob('vehicles.csv'))
        if len(csvs)>1:raise ValueError('Attach only one vehicles.csv')
        if csvs:
            with zipfile.ZipFile(archive,'w',zipfile.ZIP_STORED) as z:z.write(csvs[0],'vehicles.csv')
        elif not archive.exists():
            url='https://zenodo.org/records/10457828/files/Used%20Cars%20Dataset.zip?download=1'
            temporary=archive.with_suffix('.part')
            with requests.get(url,stream=True,timeout=(30,180)) as response:
                response.raise_for_status()
                with temporary.open('wb') as f:
                    for chunk in response.iter_content(1024*1024):f.write(chunk)
            if not zipfile.is_zipfile(temporary):raise ValueError('Dataset download was not a ZIP')
            temporary.replace(archive)
        subprocess.run([sys.executable,'-u','-m','scripts.prepare_full',str(archive),'--output',str(DATA),'--workers','4'],check=True)
cohort_hash=hashlib.sha256(DATA.read_bytes()).hexdigest()
expected=json.loads((ROOT/'models/transfer-asking/manifest.json').read_text())['cohort_sha256']
if cohort_hash!=expected:raise ValueError('This notebook compares against a frozen baseline for the original cohort. Attach the matching original full-cohort.jsonl; do not compare a different cohort against its scores.')
print('Cohort SHA256:',cohort_hash)


## Train, assess, refit

Eight-hour session budget. The trainer caches each completed configuration. Three fixed seed repetitions are averaged in dollars after validation chooses configurations/weights. The unfiltered ensemble is retained as the default unless the recorded promotion criteria are met. Full eligible-data refit occurs after the assessment.

In [ ]:
OUTPUT=ROOT/'models/robust-asking-isolation';OUTPUT.mkdir(parents=True,exist_ok=True)
cmd=[sys.executable,'-u','-m','marketbridge.robust','--cohort',str(DATA),'--structural',str(ROOT/'models/structural-transfer'),'--output',str(OUTPUT),'--isolation','auto']
status='ready' if (OUTPUT/'manifest.json').exists() else 'pending'
if status!='ready':
    remaining=max(0,8*3600-(time.monotonic()-START));log=OUTPUT/'training.log'
    if remaining>120:
        with log.open('a',buffering=1) as stream:
            process=subprocess.Popen(cmd,stdout=stream,stderr=subprocess.STDOUT,start_new_session=True)
            try:rc=process.wait(timeout=remaining)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid,signal.SIGINT)
                try:process.wait(timeout=60)
                except subprocess.TimeoutExpired:os.killpg(process.pid,signal.SIGTERM);process.wait(timeout=30)
                rc=None
        status='ready' if (OUTPUT/'manifest.json').exists() else ('failed' if rc not in (None,0) else 'pending')
        print(log.read_text(errors='replace')[-10000:])
(ROOT/'training-status.json').write_text(json.dumps({'status':status}))
print('Training status:',status)
if status=='failed':raise RuntimeError('Training failed; inspect the log above')


In [ ]:
if status=='ready':
    from marketbridge.model_storage import pack_bundle
    print('Model storage:',pack_bundle(OUTPUT))
    print((OUTPUT/'evaluation.json').read_text())
    destination=Path('/kaggle/working/Filtered_Pricing_Trained.zip')
    with zipfile.ZipFile(destination,'w',zipfile.ZIP_DEFLATED) as z:
        for file in OUTPUT.rglob('*'):
            if file.is_file() and 'search-cache' not in file.parts and file.name!='training.log':z.write(file,str(file.relative_to(ROOT)))
        for file in (ROOT/'marketbridge').glob('*.py'):z.write(file,str(file.relative_to(ROOT)))
    print('Download:',destination)
    print('Extract into the current Pricing repository. Keep its matching models/structural-transfer bundle and unfiltered ensemble.')
else:print('Save Kaggle output and attach it to a new session. Rerun unchanged to reuse completed search fits.')
